# Aircraft Maintenance Manual - Knowledge Graph Pipeline

This notebook uses `SimpleKGPipeline` from `neo4j-graphrag` to transform the A320-200 Maintenance Manual into a searchable knowledge graph with a single pipeline call. The pipeline automates:

1. **Chunking** - Splitting the document into searchable pieces
2. **Embedding** - Generating vector representations for semantic search
3. **Entity extraction** - Using an LLM to identify operating limits from the text
4. **Entity resolution** - Deduplicating extracted entities across chunks

After the pipeline runs, we create indexes for retrieval and cross-link the extracted knowledge to the aircraft topology from Lab 2.

**Prerequisites:**
- Complete **Lab 2** (Databricks ETL) to load the aircraft topology graph (Aircraft, System, Component nodes)
- Running in a Databricks notebook environment

**Learning Objectives:**
- Understand the Document -> Chunk graph structure for semantic search
- Configure and run SimpleKGPipeline for automated chunking, embedding, and entity extraction
- Create vector and fulltext indexes for retrieval
- Cross-link extracted entities to the aircraft topology (APPLIES_TO, HAS_LIMIT)
- Perform semantic similarity search over maintenance procedures

---

## Why SimpleKGPipeline?

Building a GraphRAG knowledge graph involves several steps: splitting documents into chunks, embedding them, extracting structured entities with an LLM, and resolving duplicate entities. `SimpleKGPipeline` orchestrates all of these in a single pass, producing:

```
(:Document) <-[:FROM_DOCUMENT]- (:Chunk) -[:NEXT_CHUNK]-> (:Chunk)
                                    |
                              (embedding vector)
                              (extracted entities: OperatingLimit)
```

The pipeline reads the maintenance manual, chunks it, uses an LLM to extract `OperatingLimit` entities (EGT thresholds, vibration limits, etc.), and stores everything in Neo4j with embeddings for semantic search.

## Section 1: Configuration

Enter your Neo4j Aura connection details below. You received these credentials when you created your Neo4j Aura instance in Lab 1.

In [ ]:
# ============================================
# CONFIGURATION - Enter your Neo4j credentials
# ============================================

NEO4J_URI = ""  # e.g., "neo4j+s://xxxxxxxx.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""  # Your password from Lab 1

# Unity Catalog Volume path (pre-configured by workshop admin)
DATA_PATH = "/Volumes/databricks-neo4j-lab/lab-schema/lab-volume"

# Validate configuration
if not NEO4J_URI or not NEO4J_PASSWORD:
    print("WARNING: Please enter your Neo4j credentials above before running the notebook!")
else:
    print("Configuration ready!")
    print(f"Neo4j URI: {NEO4J_URI}")
    print(f"Data Path: {DATA_PATH}")

## Setup

Import required modules and configure the environment.

In [ ]:
from neo4j_graphrag.indexes import create_vector_index, create_fulltext_index

from data_utils import (
    Neo4jConnection, VolumeDataLoader, get_embedder, get_llm,
    run_pipeline, EMBEDDING_DIMENSIONS
)

## Maintenance Manual Data

We'll load the A320-200 Maintenance and Troubleshooting Manual from the Unity Catalog Volume. This file was uploaded during lab setup along with the aircraft CSV data.

**Volume Path:** `/Volumes/databricks-neo4j-lab/lab-schema/lab-volume/MAINTENANCE_A320.md`

This comprehensive document includes:

- **Aircraft specifications** for the SkyWays A320-200 fleet (5 aircraft)
- **System architecture** covering Engines (V2500-A1), Avionics, and Hydraulics
- **Troubleshooting procedures** with fault codes and decision trees
- **Operating limits** and scheduled maintenance tasks

This realistic maintenance manual will allow semantic search queries like:
- "How do I troubleshoot engine vibration?"
- "What are the EGT limits during takeoff?"
- "What causes hydraulic pressure loss?"

In [ ]:
# Load text from the maintenance manual in Unity Catalog Volume
loader = VolumeDataLoader("MAINTENANCE_A320.md", volume_path=DATA_PATH)
MANUAL_TEXT = loader.text

# Document metadata
DOCUMENT_ID = "AMM-A320-2024-001"
AIRCRAFT_TYPE = "A320-200"

metadata = loader.get_metadata()
print(f"Loaded: {metadata['name']}")
print(f"From Volume: {metadata['volume']}")
print(f"Size: {metadata['size']:,} characters")
print(f"\nFirst 500 characters:")
print(f"{MANUAL_TEXT[:500]}...")

## Connect to Neo4j

Create a connection to your Neo4j database. This should already contain the aircraft topology from Lab 2.

In [ ]:
neo4j = Neo4jConnection(uri=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD).verify()
driver = neo4j.driver

# Show existing graph statistics
neo4j.get_graph_stats()

## Clear Previous Data (Optional)

For a clean start, remove any existing Document, Chunk, and OperatingLimit nodes from previous runs. This preserves your aircraft topology (Aircraft, System, Component nodes).

In [ ]:
neo4j.clear_chunks()

## Initialize LLM and Embedder

Set up the Large Language Model (LLM) and embedding model for the pipeline.

- **LLM**: Uses Databricks Foundation Model APIs (Llama 3.3 70B) for entity extraction
- **Embedder**: Uses Databricks Foundation Model APIs (BGE-large) for chunk embeddings

In [ ]:
llm = get_llm()
embedder = get_embedder()

print(f"LLM initialized: {llm.model_id}")
print(f"Embedder initialized: {embedder.model_id}")

---

# Part 1: SimpleKGPipeline

The `SimpleKGPipeline` from `neo4j-graphrag` handles the full transformation from raw text to a queryable knowledge graph in a single call. It:

1. **Splits** the document into chunks (800 characters with 100-character overlap)
2. **Embeds** each chunk using the BGE embedding model (1024-dimensional vectors)
3. **Extracts** `OperatingLimit` entities from each chunk using the LLM
4. **Resolves** duplicate entities (the same limit mentioned across multiple chunks becomes one node)
5. **Stores** everything in Neo4j: Document, Chunk nodes with embeddings, and OperatingLimit entities

### Chunking Trade-offs

Larger chunks give the LLM more context during entity extraction, improving its ability to resolve references like "the engine" to a specific component. Smaller chunks produce more precise retrieval results. A moderate chunk size (800 characters) with overlap (100 characters) at boundaries is a good starting point.

### Entity Extraction Schema

The pipeline is configured with a schema that tells the LLM what to extract. We define a single entity type:

- **OperatingLimit** - Aircraft operating parameter thresholds (EGT limits, vibration thresholds, etc.)
  - Properties: `name`, `parameterName`, `unit`, `regime`, `minValue`, `maxValue`, `aircraftType`

A custom extraction prompt teaches the LLM to distinguish aircraft types (A320-200) from engine types (V2500) and to use sensor parameter names (EGT, Vibration, N1Speed) that match the operational graph.

### Context-Aware Chunking

A `ContextPrependingSplitter` prepends `[DOCUMENT CONTEXT] Aircraft Type: A320-200` to every chunk. Without this, chunks deep in engine-specific sections would lack aircraft type context, causing the LLM to confuse engine designations for aircraft types.

## Run the Pipeline

Process the full maintenance manual. The pipeline will chunk the text, generate embeddings, and extract operating limit entities.

> **Note:** This cell takes 2-5 minutes depending on the LLM response time. The LLM reads each chunk and extracts any operating limits it finds.

In [ ]:
run_pipeline(
    driver=driver,
    llm=llm,
    embedder=embedder,
    text=MANUAL_TEXT,
    document_metadata={
        "documentId": DOCUMENT_ID,
        "aircraftType": AIRCRAFT_TYPE,
        "title": "A320-200 Maintenance and Troubleshooting Manual",
        "type": "maintenance_manual",
    },
    context=f"[DOCUMENT CONTEXT] Aircraft Type: {AIRCRAFT_TYPE} | Title: A320-200 Maintenance Manual\n\n",
)

## Inspect the Knowledge Graph

The pipeline created Document, Chunk, and OperatingLimit nodes. Let's verify what was built.

In [ ]:
# Updated graph statistics
neo4j.get_graph_stats()

# Document-Chunk structure
records, _, _ = driver.execute_query("""
    MATCH (d:Document)
    OPTIONAL MATCH (d)<-[:FROM_DOCUMENT]-(c:Chunk)
    RETURN d.documentId AS document_id, d.title AS title, count(c) AS chunks
""")
print("\n=== Document-Chunk Structure ===")
for record in records:
    print(f"Document: {record['document_id']}")
    print(f"  Title: {record['title']}")
    print(f"  Chunks: {record['chunks']}")

# Chunk chain sample
records, _, _ = driver.execute_query("""
    MATCH (c:Chunk)
    WHERE c.index IS NOT NULL
    OPTIONAL MATCH (c)-[:NEXT_CHUNK]->(next:Chunk)
    RETURN c.index AS idx,
           substring(c.text, 0, 80) AS text,
           next.index AS next_idx
    ORDER BY c.index
    LIMIT 5
""")
print("\n=== Chunk Chain (first 5) ===")
for record in records:
    next_str = f" -> Chunk {record['next_idx']}" if record['next_idx'] is not None else " (end)"
    print(f"Chunk {record['idx']}: \"{record['text']}...\"{next_str}")

## Inspect Extracted Entities

The LLM extracted `OperatingLimit` entities from the maintenance manual. These represent specific operating thresholds like maximum EGT temperatures, vibration limits, and fuel flow ranges.

Entity resolution has already deduplicated these -- the same limit mentioned across multiple chunks appears as a single node.

In [ ]:
records, _, _ = driver.execute_query("""
    MATCH (ol:OperatingLimit)
    RETURN ol.name AS name, ol.parameterName AS param,
           ol.aircraftType AS aircraft, ol.unit AS unit,
           ol.maxValue AS max_value, ol.regime AS regime
    ORDER BY ol.name
""")
print(f"Extracted {len(records)} OperatingLimit entities:\n")
for r in records:
    regime = f" ({r['regime']})" if r['regime'] else ""
    unit = r['unit'] or ''
    max_val = r['max_value'] or 'N/A'
    print(f"  {r['name']}: max={max_val} {unit}{regime} [{r['aircraft']}]")

---

# Part 2: Create Indexes for Retrieval

The pipeline stored embeddings on each Chunk node. Now we create indexes so retrieval queries can efficiently search them:

- **Vector index** - For semantic similarity search (cosine distance over 1024-dimensional embeddings)
- **Fulltext index** - For keyword-based search (enables `HybridRetriever` in later notebooks)

In [ ]:
INDEX_NAME = "maintenanceChunkEmbeddings"

create_vector_index(
    driver=driver,
    name=INDEX_NAME,
    label="Chunk",
    embedding_property="embedding",
    dimensions=EMBEDDING_DIMENSIONS,
    similarity_fn="cosine"
)
print(f"Created vector index: {INDEX_NAME} ({EMBEDDING_DIMENSIONS} dimensions, cosine similarity)")

In [ ]:
FULLTEXT_INDEX_NAME = "maintenanceChunkText"

create_fulltext_index(
    driver=driver,
    name=FULLTEXT_INDEX_NAME,
    label="Chunk",
    node_properties=["text"]
)
print(f"Created fulltext index: {FULLTEXT_INDEX_NAME}")

---

# Part 3: Cross-Linking to Aircraft Topology

The pipeline created Document, Chunk, and OperatingLimit nodes, but they aren't yet connected to the aircraft topology from Lab 2. Two cross-links bridge these worlds:

1. **`Document -[:APPLIES_TO]-> Aircraft`** - The Document node has an `aircraftType` property ("A320-200") that matches `Aircraft.model`. This lets retrieval queries traverse from a matched chunk through its Document to the specific aircraft it applies to.

2. **`Sensor -[:HAS_LIMIT]-> OperatingLimit`** - Extracted OperatingLimit entities have a `parameterName` (e.g. "EGT") that matches `Sensor.type`. This connects sensor nodes in the operational graph to the limits documented in the maintenance manual.

```
Chunk -[:FROM_DOCUMENT]-> Document -[:APPLIES_TO]-> Aircraft
                                                      |
                                            [:HAS_SYSTEM]-> System
                                                              |
                                                    [:HAS_SENSOR]-> Sensor
                                                                      |
                                                            [:HAS_LIMIT]-> OperatingLimit
```

In [ ]:
# Document -[:APPLIES_TO]-> Aircraft
records, _, _ = driver.execute_query("""
    MATCH (d:Document) WHERE d.aircraftType IS NOT NULL
    MATCH (a:Aircraft {model: d.aircraftType})
    MERGE (d)-[:APPLIES_TO]->(a)
    RETURN d.documentId AS doc, a.model AS aircraft, count(*) AS count
""")
print("Document -[:APPLIES_TO]-> Aircraft:")
for r in records:
    print(f"  {r['doc']} -> {r['aircraft']}")

# Sensor -[:HAS_LIMIT]-> OperatingLimit
records, _, _ = driver.execute_query("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(sys:System)-[:HAS_SENSOR]->(s:Sensor)
    MATCH (ol:OperatingLimit {parameterName: s.type, aircraftType: a.model})
    MERGE (s)-[:HAS_LIMIT]->(ol)
    RETURN s.type AS sensor, ol.name AS limit, count(*) AS count
""")
print("\nSensor -[:HAS_LIMIT]-> OperatingLimit:")
for r in records:
    print(f"  {r['sensor']} -> {r['limit']}")
if not records:
    print("  (No matches — depends on which limits the LLM extracted)")

In [ ]:
# Verify the full traversal path: Chunk -> Document -> Aircraft -> System -> Sensor -> OperatingLimit
records, _, _ = driver.execute_query("""
    MATCH (c:Chunk)-[:FROM_DOCUMENT]->(d:Document)-[:APPLIES_TO]->(a:Aircraft)
    WITH a, count(c) AS chunks
    OPTIONAL MATCH (a)-[:HAS_SYSTEM]->(sys:System)-[:HAS_SENSOR]->(s:Sensor)-[:HAS_LIMIT]->(ol:OperatingLimit)
    RETURN a.tail_number AS aircraft, a.model AS model, chunks,
           count(DISTINCT ol) AS operating_limits
    ORDER BY a.tail_number
""")
print("Full graph traversal (Chunk -> Document -> Aircraft -> ... -> OperatingLimit):")
print(f"{'Aircraft':<12} {'Model':<12} {'Chunks':<8} {'Limits':<8}")
print("-" * 40)
for r in records:
    print(f"{r['aircraft']:<12} {r['model']:<12} {r['chunks']:<8} {r['operating_limits']:<8}")

---

# Part 4: Test Semantic Search

With embeddings stored and the vector index created, we can now search for chunks that are semantically similar to a query. The search:
1. Converts the query to an embedding vector
2. Finds chunks with similar embedding vectors using cosine distance
3. Returns results ranked by similarity score

In [ ]:
def vector_search(driver, embedder, query: str, top_k: int = 3):
    """Search for chunks similar to the query."""
    query_embedding = embedder.embed_query(query)

    records, _, _ = driver.execute_query("""
        CALL db.index.vector.queryNodes($index_name, $top_k, $embedding)
        YIELD node, score
        RETURN node.text as text, node.index as idx, score
    """, index_name=INDEX_NAME, top_k=top_k, embedding=query_embedding)

    return records

# Test search with a maintenance query
query = "How do I troubleshoot engine vibration?"
print(f"Query: \"{query}\"\n")
print("=" * 70)

results = vector_search(driver, embedder, query)
for i, record in enumerate(results):
    print(f"\n[{i+1}] Score: {record['score']:.4f} (Chunk {record['idx']})")
    print(f"    {record['text'][:250]}...")

## Compare Different Queries

Try different maintenance queries to see how semantic search finds relevant procedures even with different wording.

In [ ]:
queries = [
    "What are the EGT limits during takeoff?",
    "How to detect bearing wear in the engine?",
    "What causes hydraulic pressure loss?",
    "When should I replace the fuel filter?"
]

for query in queries:
    print(f"\nQuery: \"{query}\"")
    print("-" * 60)
    results = vector_search(driver, embedder, query, top_k=1)
    if results:
        record = results[0]
        print(f"Best match (score: {record['score']:.4f}):")
        print(f"  {record['text'][:200]}...")

## Summary

In this notebook, you built a complete GraphRAG knowledge graph from the A320-200 Maintenance Manual:

**Part 1 - SimpleKGPipeline:**
1. **Document-Chunk structure** - The manual was split into searchable chunks with `FROM_DOCUMENT` and `NEXT_CHUNK` relationships
2. **Embeddings** - 1024-dimensional BGE vectors capture semantic meaning of each chunk
3. **Entity extraction** - The LLM identified `OperatingLimit` entities (EGT thresholds, vibration limits, etc.)
4. **Entity resolution** - Duplicate entities across chunks were merged

**Part 2 - Indexes:**
5. **Vector index** - Cosine similarity search over chunk embeddings
6. **Fulltext index** - Keyword-based search for hybrid retrieval

**Part 3 - Cross-links:**
7. **APPLIES_TO** - Documents linked to the aircraft they describe
8. **HAS_LIMIT** - Sensors linked to their operating limits from the manual

**Part 4 - Semantic search:**
9. **Vector similarity search** - Finding maintenance procedures by meaning

Your knowledge graph now combines:
- **Structured data**: Aircraft -> System -> Component hierarchy from Lab 2
- **Unstructured data**: Maintenance manual chunks with embeddings
- **Extracted entities**: Operating limits cross-linked to sensors

---

**Next:** [GraphRAG Retrievers](04_graphrag_retrievers.ipynb) - Learn to use retrievers that combine vector search with graph traversal for context-aware answers about aircraft maintenance.

**Optional:** [Hybrid Retrievers](05_hybrid_retrievers.ipynb) - Explore hybrid search that combines vector similarity with keyword matching.

In [ ]:
# Cleanup
neo4j.close()